# PKLot: GitHub → module Python → Colab

Chỉ upload notebook này lên Colab. Push `parking_module.py` lên repo GitHub public, điền OWNER/REPO/REF bên dưới rồi Run All. REF có thể là nhánh, tag hoặc commit SHA. Module chứa loader, contracts, audit, giải nén và metrics; notebook chỉ cấu hình và gọi hàm.


In [1]:
# Điền thông tin repo chứa parking_module.py trước khi Run All.
OWNER = "tuan314159265"
REPO = "parking-occupancy"
REF = "main"  # Đổi thành master nếu cần; dùng commit SHA để tái lập.
MODULE_PATH = "parking_module.py"

from pathlib import Path
USE_GOOGLE_DRIVE = True
DRIVE_PROJECT_DIR = "parking-occupancy"
DOWNLOAD_DATA = True  # False: archive phải có sẵn và hoàn chỉnh.
REUSE_EXTRACTED = True  # Dùng lại dữ liệu đã giải nén thành công.


In [2]:
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")  # Chọn tài khoản Drive của bạn khi Colab yêu cầu.
    WORK_DIR = Path("/content/drive/MyDrive") / DRIVE_PROJECT_DIR
else:
    WORK_DIR = Path("/content/parking-occupancy")
RAW_DIR = WORK_DIR / "data/raw"
DATASET_DIR = RAW_DIR / "PKLot"
OUTPUT_DIR = WORK_DIR / "outputs/full"
WORK_DIR.mkdir(parents=True, exist_ok=True)
print("Thư mục lưu:", WORK_DIR)


ValueError: mount failed

In [ ]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "Pillow>=10,<13"], check=True)


In [ ]:
import hashlib, importlib.util, json, urllib.request
from urllib.parse import quote
assert OWNER != "YOUR_GITHUB_USERNAME", "Điền OWNER, REPO và REF ở cell cấu hình"
WORK_DIR.mkdir(parents=True, exist_ok=True)
url = (f"https://raw.githubusercontent.com/{quote(OWNER, safe='')}/"
       f"{quote(REPO, safe='')}/{quote(REF, safe='')}/{quote(MODULE_PATH, safe='/')}")
# Tải lại module mỗi lần Run All để lấy đúng REF đã cấu hình.
with urllib.request.urlopen(url, timeout=60) as response:
    code = response.read()
compile(code, MODULE_PATH, "exec")
module_file = WORK_DIR / "parking_module.py"
module_file.write_bytes(code)
spec = importlib.util.spec_from_file_location("parking_module", module_file)
parking = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = parking
spec.loader.exec_module(parking)
provenance = {"url": url, "ref": REF, "sha256": hashlib.sha256(code).hexdigest()}
(WORK_DIR / "module_source.json").write_text(json.dumps(provenance, indent=2))
print("Loaded module:", provenance)


## Tải dữ liệu thật và kiểm tra

Mặc định archive, ảnh/XML đã giải nén và kết quả nằm trên Google Drive của bạn, trong MyDrive/parking-occupancy/. Dữ liệu được giữ qua các phiên Colab. Archive khoảng 4,9 GB; cần khoảng 10–15 GB trống. Lần sau tự dùng lại dữ liệu đã chuẩn bị thành công và chạy lại audit. Nguồn UFPR có thể tải chậm; giải nén/đọc nhiều file trên Drive cũng có thể mất thời gian.


In [ ]:
# provenance.json chỉ được ghi khi lần giải nén/audit trước đã hoàn tất.
prepared_before = (OUTPUT_DIR / "provenance.json").exists() and DATASET_DIR.is_dir()
if REUSE_EXTRACTED and prepared_before:
    print("Dùng lại dữ liệu đã giải nén; chạy lại audit.")
    summary = parking.audit(root=DATASET_DIR, output=OUTPUT_DIR)
else:
    summary = parking.prepare(raw=RAW_DIR, output=OUTPUT_DIR, download=DOWNLOAD_DATA)
print(json.dumps(summary, indent=2))
assert summary["valid_frames"] > 0


In [ ]:
# Đọc một mẫu hợp lệ trực tiếp bằng module lấy từ GitHub.
with (OUTPUT_DIR / "valid_samples.jsonl").open() as stream:
    sample = json.loads(next(stream))
frame = parking.read_frame(RAW_DIR / "PKLot" / sample["xml_path"], RAW_DIR / "PKLot")
print("Sample:", frame.sample_id)
print("Camera / weather / date:", frame.camera, frame.weather, frame.date)
print("Parking spaces:", len(frame.spaces))
print("Outputs:", OUTPUT_DIR)


## Nối bước tiếp theo

Gọi trực tiếp `parking.VehicleDetector`, `parking.OccupancyPrediction`, `parking.occupancy_metrics` khi triển khai detector và ghép ô. Polygon XML là vùng ô đỗ, không phải hộp xe. Chưa triển khai mô hình trong notebook này.
